# Notebook 2: Preprocessing & Feature Engineering
## Medical Appointment No-Show Prediction & Demand Forecasting

This notebook covers:
1. Missing value imputation strategies
2. Feature engineering (temporal, interaction, derived features)
3. Categorical encoding pipeline
4. Train-test split with class imbalance handling (SMOTE / class weights)
5. Daily demand aggregation for time series forecasting
6. Saving preprocessed data for model training notebooks

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
import joblib
import os
import sys
import warnings
warnings.filterwarnings('ignore')

sys.path.append(os.path.abspath('..'))
from src.preprocessing import load_data

# Load raw data (applies column renaming automatically)
df = load_data()
print(f'Missing values: {df.isnull().sum().sum():,} total')
df.head(3)

Dataset loaded: 109593 rows × 26 columns
Missing values: 80,592 total


,specialty,appointment_time,gender,no_show,disability,place,appointment_shift,age,under_12,over_60,...,storm_day_before,rain_intensity,heat_intensity,appointment_date_continuous,Hipertension,Diabetes,Alcoholism,Handcap,Scholarship,SMSreceived
0,psychotherapy,17,F,yes,intellectual,Lake Marvinville,afternoon,9.0,1,0,...,1,no_rain,warm,2020-01-01,0,0,0,0,0,0
1,NaN,7,M,no,intellectual,ITAPEMA,morning,11.0,1,0,...,1,no_rain,cold,2020-01-01,0,0,0,0,0,0
2,speech therapy,16,M,no,intellectual,ITAJAÍ,afternoon,8.0,1,0,...,1,no_rain,warm,2020-01-01,0,0,0,0,0,0


## Section 1: Missing Value Imputation

**Strategy:**
| Column | Missing % | Strategy |
|--------|-----------|----------|
| `age` | ~21% | Median imputation |
| `specialty` | ~18% | Fill with 'Unknown' category |
| `disability` | ~15% | Fill with 'Unknown' (categorical: intellectual, motor) |
| `place` | ~10.5% | Fill with 'Unknown' category |
| Weather cols | ~2% | Median for numeric, mode for categorical |

In [2]:
# ── Missing Value Imputation ──────────────────────────────────────────────────
print('BEFORE imputation:')
print(df.isnull().sum()[df.isnull().sum() > 0])
print(f'\nTotal missing: {df.isnull().sum().sum():,}')

# Age: median imputation
if 'age' in df.columns and df['age'].isnull().sum() > 0:
    age_median = df['age'].median()
    df['age'] = df['age'].fillna(age_median)
    print(f'\n✓ age: filled with median = {age_median}')

# Specialty: Unknown category
if 'specialty' in df.columns and df['specialty'].isnull().sum() > 0:
    df['specialty'] = df['specialty'].fillna('Unknown')
    print('✓ specialty: filled with "Unknown"')

# Disability: fill with 'Unknown' (categorical column: intellectual, motor)
if 'disability' in df.columns and df['disability'].isnull().sum() > 0:
    df['disability'] = df['disability'].fillna('Unknown')
    print('✓ disability: filled with "Unknown"')

# Place: Unknown category
if 'place' in df.columns and df['place'].isnull().sum() > 0:
    df['place'] = df['place'].fillna('Unknown')
    print('✓ place: filled with "Unknown"')

# Weather numeric: median
for col in ['avg_temp', 'max_temp', 'rain', 'max_rain']:
    if col in df.columns and df[col].isnull().sum() > 0:
        med = df[col].median()
        df[col] = df[col].fillna(med)
        print(f'✓ {col}: filled with median = {med:.2f}')

# Weather categorical: mode
for col in ['heat_intensity', 'rain_intensity']:
    if col in df.columns and df[col].isnull().sum() > 0:
        mode_val = df[col].mode()[0]
        df[col] = df[col].fillna(mode_val)
        print(f'✓ {col}: filled with mode = "{mode_val}"')

# Binary weather: fill with 0
for col in ['rainy_day_before', 'storm_day_before']:
    if col in df.columns and df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(0)
        print(f'✓ {col}: filled with 0')

# Fill any remaining NaN
for col in df.columns:
    if df[col].isnull().sum() > 0:
        if df[col].dtype in ['float64', 'int64']:
            df[col] = df[col].fillna(df[col].median())
        else:
            df[col] = df[col].fillna(df[col].mode()[0] if len(df[col].mode()) > 0 else 'Unknown')
        print(f'✓ {col}: filled remaining NaN')

print(f'\nAFTER imputation: {df.isnull().sum().sum()} missing values')
print(f'Dataset shape: {df.shape}')

BEFORE imputation:
specialty     20127
disability    17020
place         11539
age           22960
avg_temp       2211
rain           2245
max_temp       2227
max_rain       2263
dtype: int64

Total missing: 80,592

✓ age: filled with median = 12.0
✓ specialty: filled with "Unknown"
✓ disability: filled with "Unknown"
✓ place: filled with "Unknown"
✓ avg_temp: filled with median = 20.60
✓ max_temp: filled with median = 23.90
✓ rain: filled with median = 0.01
✓ max_rain: filled with median = 0.20

AFTER imputation: 0 missing values
Dataset shape: (109593, 26)


## Section 2: Feature Engineering

Creating new features:
- **Temporal:** day_of_week, month, is_weekend, is_month_start/end, week_of_year
- **Age-based:** age_group bins (child, teen, young_adult, adult, middle_aged, senior)
- **Health composite:** chronic_condition_count (sum of hypertension + diabetes + alcoholism + handicap)
- **Appointment shift:** ordinal encoding
- **Weather interaction:** temp_rain_interaction
- **Patient interaction:** companion_disability

In [3]:
# ── Feature Engineering ───────────────────────────────────────────────────────

# 1. Temporal features from appointment_date_continuous
if 'appointment_date_continuous' in df.columns:
    df['appointment_date'] = pd.to_datetime(df['appointment_date_continuous'], errors='coerce')
    df['day_of_week'] = df['appointment_date'].dt.dayofweek       # 0=Mon, 6=Sun
    df['month'] = df['appointment_date'].dt.month
    df['day_of_month'] = df['appointment_date'].dt.day
    df['week_of_year'] = df['appointment_date'].dt.isocalendar().week.astype(int)
    df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)
    df['is_monday'] = (df['day_of_week'] == 0).astype(int)
    df['is_month_start'] = df['appointment_date'].dt.is_month_start.astype(int)
    df['is_month_end'] = df['appointment_date'].dt.is_month_end.astype(int)
    print('✓ Temporal features created: day_of_week, month, day_of_month, week_of_year, is_weekend, is_monday, is_month_start, is_month_end')

# 2. Age group bins
if 'age' in df.columns:
    df['age_group'] = pd.cut(df['age'], bins=[0, 12, 18, 35, 50, 65, 120],
                              labels=['child', 'teen', 'young_adult', 'adult', 'middle_aged', 'senior'],
                              right=False)
    df['age_group'] = df['age_group'].astype(str)
    print('✓ age_group feature created')

# 3. Health risk composite score
health_cols = ['Hipertension', 'Diabetes', 'Alcoholism', 'Handcap']
existing_health = [c for c in health_cols if c in df.columns]
if existing_health:
    df['chronic_condition_count'] = df[existing_health].sum(axis=1)
    print(f'✓ chronic_condition_count created from {existing_health}')

# 4. Appointment shift ordinal encoding
if 'appointment_shift' in df.columns:
    shift_map = {'morning': 0, 'afternoon': 1, 'evening': 2, 'night': 3}
    df['shift_encoded'] = df['appointment_shift'].str.lower().map(shift_map).fillna(0).astype(int)
    print('✓ shift_encoded feature created')

# 5. Weather interaction
if 'avg_temp' in df.columns and 'rain' in df.columns:
    df['temp_rain_interaction'] = df['avg_temp'] * df['rain']
    print('✓ temp_rain_interaction feature created')

# 6. Companion-has_disability interaction (disability is categorical)
if 'needs_companion' in df.columns and 'disability' in df.columns:
    df['has_disability'] = (df['disability'] != 'Unknown').astype(int)
    df['companion_disability'] = df['needs_companion'] * df['has_disability']
    print('✓ has_disability and companion_disability features created')

print(f'\nDataset after feature engineering: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'New columns: {df.shape[1] - 26} added')
df.head(3)

✓ Temporal features created: day_of_week, month, day_of_month, week_of_year, is_weekend, is_monday, is_month_start, is_month_end
✓ age_group feature created
✓ chronic_condition_count created from ['Hipertension', 'Diabetes', 'Alcoholism', 'Handcap']
✓ shift_encoded feature created
✓ temp_rain_interaction feature created
✓ has_disability and companion_disability features created

Dataset after feature engineering: 109,593 rows × 41 columns
New columns: 15 added


,specialty,appointment_time,gender,no_show,disability,place,appointment_shift,age,under_12,over_60,...,is_weekend,is_monday,is_month_start,is_month_end,age_group,chronic_condition_count,shift_encoded,temp_rain_interaction,has_disability,companion_disability
0,psychotherapy,17,F,yes,intellectual,Lake Marvinville,afternoon,9.0,1,0,...,0,0,1,0,child,0,1,0.0000,1,1
1,Unknown,7,M,no,intellectual,ITAPEMA,morning,11.0,1,0,...,0,0,1,0,child,0,0,0.2862,1,1
2,speech therapy,16,M,no,intellectual,ITAJAÍ,afternoon,8.0,1,0,...,0,0,1,0,child,0,1,0.2161,1,1


## Section 3: Categorical Encoding & Target Variable Encoding

In [ ]:
# ── Encode Target Variable ────────────────────────────────────────────────────
# Convert no_show to numeric (handles pandas 3.x Arrow-backed strings)
df['no_show'] = np.where(df['no_show'].astype(str).str.lower() == 'yes', 1, 0)
print('Target variable encoded: yes=1 (no-show), no=0 (showed up)')
print(f'No-show rate: {df["no_show"].mean():.3f}')
print(f'no_show dtype: {df["no_show"].dtype}')

# ── Encode Categorical Features ──────────────────────────────────────────────
categorical_cols = ['gender', 'specialty', 'appointment_shift', 'place',
                    'heat_intensity', 'rain_intensity', 'disability', 'age_group']
categorical_cols = [c for c in categorical_cols if c in df.columns]

label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    df[col + '_encoded'] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le
    print(f'✓ {col}: {len(le.classes_)} categories encoded')

print(f'\nLabel encoders saved for {len(label_encoders)} columns')
print(f'Dataset shape: {df.shape}')

Target variable encoded: yes=1 (no-show), no=0 (showed up)


TypeError: Cannot perform reduction 'mean' with string dtype

## Section 4: Prepare Feature Matrix for Classification

Select features, handle remaining non-numeric columns, and create train/test split with class imbalance handling.

In [ ]:
# ── Define Feature Set for Classification ─────────────────────────────────────
# Drop non-feature columns (raw text, dates, original categorical before encoding)
drop_cols = ['no_show', 'appointment_date_continuous', 'appointment_date']
drop_cols += [c for c in categorical_cols if c in df.columns]  # drop original text columns

feature_cols = [c for c in df.columns if c not in drop_cols]
# Keep only numeric features
X = df[feature_cols].select_dtypes(include=[np.number])
y = df['no_show']

print(f'Feature matrix: {X.shape}')
print(f'Target distribution:\n{y.value_counts()}')
print(f'\nFeatures used ({len(X.columns)}):')
print(list(X.columns))

In [ ]:
# ── Train-Test Split (Stratified) ─────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train set: {X_train.shape[0]:,} samples')
print(f'Test set:  {X_test.shape[0]:,} samples')
print(f'Train no-show rate: {y_train.mean():.3f}')
print(f'Test no-show rate:  {y_test.mean():.3f}')

# ── SMOTE for Class Imbalance ────────────────────────────────────────────────
print('\n--- Applying SMOTE on Training Data ---')
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)
print(f'Before SMOTE: {y_train.value_counts().to_dict()}')
print(f'After SMOTE:  {pd.Series(y_train_smote).value_counts().to_dict()}')
print(f'SMOTE train shape: {X_train_smote.shape}')

# ── Standard Scaling ─────────────────────────────────────────────────────────
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)
X_train_smote_scaled = pd.DataFrame(scaler.transform(X_train_smote), columns=X_train.columns)
print(f'\n✓ Standard scaling applied')

## Section 5: Daily Demand Aggregation for Time Series Forecasting

Aggregate appointment-level data to daily counts for demand forecasting.  
Create lag features, rolling averages, and temporal features for regression models.

In [ ]:
# ── Daily Demand Aggregation ──────────────────────────────────────────────────
if 'appointment_date' not in df.columns:
    df['appointment_date'] = pd.to_datetime(df['appointment_date_continuous'], errors='coerce')

# Aggregate to daily level
daily = df.groupby('appointment_date').agg(
    total_appointments=('appointment_date', 'size'),
    no_show_count=('no_show', 'sum'),
    avg_age=('age', 'mean'),
    avg_temp=('avg_temp', 'mean'),
    max_temp=('max_temp', 'mean'),
    avg_rain=('rain', 'mean'),
).reset_index()

# Derived daily features
daily['no_show_rate'] = daily['no_show_count'] / daily['total_appointments']
daily['show_count'] = daily['total_appointments'] - daily['no_show_count']
daily['day_of_week'] = daily['appointment_date'].dt.dayofweek
daily['month'] = daily['appointment_date'].dt.month
daily['is_weekend'] = (daily['day_of_week'] >= 5).astype(int)
daily['week_of_year'] = daily['appointment_date'].dt.isocalendar().week.astype(int)

# Lag features for demand
for lag in [1, 2, 3, 7, 14]:
    daily[f'demand_lag_{lag}'] = daily['total_appointments'].shift(lag)

# Rolling averages
daily['demand_rolling_7'] = daily['total_appointments'].rolling(7).mean()
daily['demand_rolling_14'] = daily['total_appointments'].rolling(14).mean()
daily['demand_rolling_30'] = daily['total_appointments'].rolling(30).mean()

# Drop rows with NaN from lag/rolling features
daily_clean = daily.dropna().reset_index(drop=True)

print(f'Daily demand dataset: {daily_clean.shape[0]} days × {daily_clean.shape[1]} features')
print(f'Average daily appointments: {daily_clean["total_appointments"].mean():.1f}')
print(f'Date range: {daily_clean["appointment_date"].min()} to {daily_clean["appointment_date"].max()}')
daily_clean.head()

In [ ]:
# ── Save Preprocessed Data & Artifacts ────────────────────────────────────────
os.makedirs('../data', exist_ok=True)
os.makedirs('../models', exist_ok=True)

# Save classification-ready data
X_train.to_csv('../data/X_train.csv', index=False)
X_test.to_csv('../data/X_test.csv', index=False)
y_train.to_csv('../data/y_train.csv', index=False)
y_test.to_csv('../data/y_test.csv', index=False)
X_train_smote.to_csv('../data/X_train_smote.csv', index=False)
pd.Series(y_train_smote, name='no_show').to_csv('../data/y_train_smote.csv', index=False)

# Save daily demand data
daily_clean.to_csv('../data/daily_demand.csv', index=False)

# Save full processed dataframe
df.to_csv('../data/processed_data.csv', index=False)

# Save preprocessing artifacts
joblib.dump(label_encoders, '../models/label_encoders.joblib')
joblib.dump(scaler, '../models/scaler.joblib')
joblib.dump(list(X.columns), '../models/feature_columns.joblib')

print('✓ Preprocessed data saved to data/ folder')
print('✓ Label encoders, scaler, and feature columns saved to models/ folder')
print(f'\nFiles saved:')
for f in ['X_train.csv', 'X_test.csv', 'y_train.csv', 'y_test.csv',
          'X_train_smote.csv', 'y_train_smote.csv', 'daily_demand.csv', 'processed_data.csv']:
    print(f'  data/{f}')
for f in ['label_encoders.joblib', 'scaler.joblib', 'feature_columns.joblib']:
    print(f'  models/{f}')